In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import awkward as ak
import hist
import matplotlib as mpl
import matplotlib.pyplot as plt
import mplhep
import numpy as np
import pandas as pd
import uproot

from analysis.utils.geometry import Geometry, get_rebinned_geometry
from analysis.utils.plot_utils import setup
from analysis.utils.truth import get_truth_particle
from analysis.utils.units import um
from analysis.utils.utils import (
    get_color_from_pdg,
    get_label_from_pdg,
    get_parquet_path,
    get_root_path,
)

In [3]:
setup()

In [4]:
def get_rebinned_df(
    hits_df: pd.DataFrame,
    old_geometry: Geometry,
    new_geometry: Geometry,
    layer_var: str = "hit_layerID",
    pixel_x_var: str = "hit_colID",
    pixel_y_var: str = "hit_rowID",
    energy_var: str = "hit_edep",
) -> pd.DataFrame:
    """
    Get hits with different bin size.
    This is useful to cluster pixels and reduce total number of pixels.
    """
    x_bin_factor = new_geometry.pixel_x_size / old_geometry.pixel_x_size
    y_bin_factor = new_geometry.pixel_y_size / old_geometry.pixel_y_size
    hits_resampled = hits_df.copy()
    hits_resampled.loc[:, "new_pixel_x"] = (
        hits_df[pixel_x_var].values // x_bin_factor
    ).astype(int)
    hits_resampled.loc[:, "new_pixel_y"] = (
        hits_df[pixel_y_var].values // y_bin_factor
    ).astype(int)

    hits_resampled["from_muon"] = 0
    hits_resampled.loc[hits_resampled["hit_pdgc"].abs() == 13, "from_muon"] = 1

    hits_resampled["from_electron"] = 0
    hits_resampled.loc[hits_resampled["hit_pdgc"].abs() == 11, "from_electron"] = 1

    aggregator_dict = {
        "energy": (energy_var, "sum"),
        "n_hits": (energy_var, "count"),
        "from_muon": ("from_muon", "any"),
        "from_electron": ("from_electron", "any"),
    }
    # if "from_muon" in hits_df.columns:
    #     aggregator_dict["from_muon"] = ("from_muon", "any")
    # if "from_electron" in hits_df.columns:
    #     aggregator_dict["from_electron"] = ("from_electron", "any")

    resampled = (
        hits_resampled.groupby(["event_id", layer_var, "new_pixel_x", "new_pixel_y"])
        .agg(**aggregator_dict)
        .reset_index()
    )

    # if "from_muon" in resampled.columns and "from_electron" in resampled.columns:
    resampled["hit_label"] = 0
    resampled.loc[resampled["from_muon"] == 1, "hit_label"] = 1
    resampled.loc[resampled["from_electron"] == 1, "hit_label"] = 2

    resampled.rename(
        columns={"new_pixel_x": "pixel_x", "new_pixel_y": "pixel_y"}, inplace=True
    )

    columns = [
        "event_id",
        "hit_layerID",
        "pixel_x",
        "pixel_y",
        "hit_label",
        "from_muon",
        "from_electron",
    ]
    # "energy", "n_hits",

    return resampled[columns]

In [5]:
path = Path("/Users/tboeckh/tmp")
run = 10000
chunk = 0
root_file = uproot.open(path / f"{run:05d}_{chunk:03d}.root")

In [8]:
primaries_df = root_file["primaries"].arrays(library="pd")

In [12]:
primaries_df.query("trackID == 1")

,evtID,vtxID,PDG,trackID,barcode,mass,charge,Vx,Vy,Vz,Vt,Px,Py,Pz,E,KE,Eta,Phi,Pt,P
0,0,0,-11,1,0,0.511047,1.0,-121.729576,1.745562,44.329330,0.0,2054.030762,4124.554199,4.124178e+05,4.124435e+05,4.124430e+05,5.187484,1.108750,4607.709961,4.124435e+05
41,1,0,11,1,0,0.511047,-1.0,3.115182,-47.711235,44.435764,0.0,6640.923340,-8392.385742,5.418054e+05,5.419111e+05,5.419106e+05,4.617716,-0.901381,10702.055664,5.419111e+05
52,2,0,11,1,0,0.510997,-1.0,-102.917542,-18.810741,42.000179,0.0,326.489166,1725.537964,7.586342e+04,7.588375e+04,7.588324e+04,4.459090,1.383797,1756.153931,7.588374e+04
84,3,0,-11,1,0,0.510988,1.0,126.903687,80.459282,43.117920,0.0,3508.545898,-3633.249268,3.874820e+05,3.875149e+05,3.875144e+05,5.033316,-0.802857,5050.781738,3.875149e+05
136,4,0,11,1,0,0.510151,-1.0,-40.851963,-27.182478,40.072788,0.0,-4829.060547,932.758362,1.491930e+06,1.491938e+06,1.491938e+06,6.408009,2.950787,4918.319336,1.491938e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26978,995,0,-11,1,0,0.511167,1.0,-63.033039,-58.861305,41.298717,0.0,6227.023926,11503.946289,5.752898e+05,5.754384e+05,5.754379e+05,4.476978,1.074661,13081.154297,5.754384e+05
26997,996,0,11,1,0,0.511018,-1.0,69.465942,-3.902829,41.275356,0.0,3891.801270,2101.517090,2.504531e+05,2.504922e+05,2.504916e+05,4.729690,0.495122,4422.950684,2.504922e+05
27010,997,0,11,1,0,0.511047,-1.0,-83.429359,17.883606,41.624477,0.0,-1840.316162,-6865.226562,4.513366e+05,4.513925e+05,4.513920e+05,4.844257,-1.832702,7107.608398,4.513925e+05
27080,998,0,11,1,0,0.510999,-1.0,-87.252510,15.584373,43.486660,0.0,2094.070801,412.250610,1.152131e+05,1.152329e+05,1.152324e+05,4.681895,0.194380,2134.264160,1.152329e+05


In [13]:
columns = [
    "event_id",
    "hit_rowID",
    "hit_colID",
    "hit_layerID",
    "hit_pdgc",
    # "hit_edep",
    "hit_trackID",
    "hit_parentID",
    "hit_fromPrimaryLepton",
]
df: pd.DataFrame = ak.to_dataframe(
    root_file["Hits/pixelHits"].arrays(columns, library="ak"),
    how="outer",
)


In [14]:
event_df = df.query("event_id == 0")

In [17]:
event_df.query("hit_trackID == 1")

event_id  hit_rowID  hit_colID  hit_layerID    hit_pdgc  \
entry subentry                                                            
0     7021             0       4377        543            4  4294967285   
      7022             0       4382        546            5  4294967285   
      7023             0       4383        547            6  4294967285   
      7024             0       4370        543            7  4294967285   
      7025             0       4337        540            8  4294967285   
      7026             0       4291        517            9  4294967285   

                hit_trackID  hit_parentID  hit_fromPrimaryLepton  
entry subentry                                                    
0     7021                1             0                   True  
      7022                1             0                   True  
      7023                1             0                   True  
      7024                1             0                   True  
      7025                1             0                   True  
      7026                1             0                   True

In [19]:
my_dict = {"color": "red"}
plot_dict = {"color": "black", "s": 5}
plot_dict.update(my_dict)
plot_dict

{'color': 'red', 's': 5}

In [ ]:
event_id = 4000
event_df = df.query("event_id == @event_id")
event_truth_df = truth_df.query("event_id == @event_id")
truth = get_truth_particle(event_truth_df)

# event_df.loc[:, "color"] = event_df["hit_pdgc"].apply(get_color_from_pdg)
event_df["color"] = "black"
event_df.loc[event_df["hit_fromPrimaryLepton"] == 1, "color"] = "red"


fig, ax = plt.subplots(ncols=3, figsize=(21, 5))

# zx
ax[0].scatter(event_df["z"], event_df["x"], c=event_df["color"], marker=".", s=1)
ax[0].scatter(
    event_truth_df["vz"],
    event_truth_df["vx"],
    c="red",
    marker="x",
    label="truth Vertex",
    s=10,
)
ax[0].set_xlabel(r"$z$ [mm]")
ax[0].set_ylabel(r"$x$ [mm]")

# zy
ax[1].scatter(event_df["z"], event_df["y"], c=event_df["color"], marker=".", s=1)
ax[1].scatter(
    event_truth_df["vz"],
    event_truth_df["vy"],
    c="red",
    marker="x",
    label="truth Vertex",
    s=10,
)
ax[1].set_xlabel(r"$z$ [mm]")
ax[1].set_ylabel(r"$y$ [mm]")

# xy
ax[2].scatter(event_df["x"], event_df["y"], c=event_df["color"], marker=".", s=1)
ax[2].scatter(
    event_truth_df["vx"],
    event_truth_df["vy"],
    c="red",
    marker="x",
    label="truth Vertex",
    s=10,
)
ax[2].set_xlabel(r"$x$ [mm]")
ax[2].set_ylabel(r"$y$ [mm]")

fig.suptitle(truth.get_latex_title())


# for i in event_df["hit_pdgc"].unique():
#     color = get_color_from_pdg(i)
#     label = get_label_from_pdg(i)
#     if color != "white":
#         ax[0].scatter(
#             [],
#             [],
#             c=color,
#             label=f"${label}$",
#             s=10,
#         )
# ax[0].legend()

plt.tight_layout()
plt.show()

In [ ]:
path = Path("/Users/tboeckh/tmp/pinpoint.root")
run = 10001
chunk = 0
root_file = uproot.open(path)

# geometry
geom_columns = ["pixel_Xpos", "pixel_Ypos", "pixel_Zpos"]
geom = root_file["geometry"].arrays(geom_columns, library="np")

# truth
truth_columns = ["evtID", "initPDG", "initX", "initY", "initZ", "initE"]
truth_df = root_file["event"].arrays(truth_columns, library="pd")
truth_df.rename(
    columns={
        "evtID": "event_id",
        "initPDG": "pdg_nu",
        "initE": "E_nu",
        "initX": "vx",
        "initY": "vy",
        "initZ": "vz",
    },
    inplace=True,
)

# truth lepton
primaries_columns = ["evtID", "trackID", "PDG", "E", "Px", "Py", "Pz"]
primaries_df = root_file["primaries"].arrays(primaries_columns, library="pd")
primaries_df.query("trackID == 1", inplace=True)
primaries_df = primaries_df[[i for i in primaries_df.columns if i != "trackID"]]
primaries_df = primaries_df.rename(
    columns={
        "evtID": "event_id",
        "E": "E_lepton",
        "PDG": "pdg_lepton",
        "Px": "px_lepton",
        "Py": "py_lepton",
        "Pz": "pz_lepton",
    }
)

if len(truth_df) != len(primaries_df):
    raise ValueError("Dataframes should have same length!")

truth_df = pd.merge(truth_df, primaries_df, left_on="event_id", right_on="event_id")

# pixel hits
columns = [
    "event_id",
    "hit_rowID",
    "hit_colID",
    "hit_layerID",
    "hit_pdgc",
    "hit_edep",
    "hit_fromPrimaryLepton",
]
df: pd.DataFrame = ak.to_dataframe(
    root_file["Hits/pixelHits"].arrays(columns, library="ak"),
    how="outer",
)

# remove hits that are outside of the detector range
max_row_index = 8596
max_col_index = 12788
df.query(f"hit_rowID < {max_row_index} and hit_colID < {max_col_index}", inplace=True)

# map pixel indices to positions
df.loc[:, "x"] = geom["pixel_Xpos"][0][df["hit_colID"].astype(int).values]
df.loc[:, "y"] = geom["pixel_Ypos"][0][df["hit_rowID"].astype(int).values]
df.loc[:, "z"] = geom["pixel_Zpos"][0][df["hit_layerID"].astype(int).values]